In [1]:
import testing_utils as testing_utils
import os

os.environ["REGION"] = "us-west-2"

In [27]:
from helpers.ticket_helper import TicketHelper
import random
from models.ticket import TicketModel
from helpers.entity_ref import EntityRefHelper
from models.user_profile import UserProfile, ProfileColorOptions
from pynamodb.exceptions import DoesNotExist
from aws_lambda_powertools import Logger
import random
from helpers.user_profile_helper import UserProfileHelper
from helpers.family_helper import FamilyHelper
from helpers.family_membership_helper import FamilyMembershipHelper
from helpers.group_membership_helper import GroupMembershipHelper
from helpers.group_helper import GroupHelper
from helpers.queue_helper import QueueHelper
import boto3
import uuid
from exceptions.membership_exceptions import MembershipAlreadyExistsAsMember
import json
import re
from openai import OpenAI
from helpers.ticket_helper import TicketHelper
from helpers.ticket_comment_helper import TicketCommentHelper
import time

In [3]:
def create_super_hero_accounts(
    model: str = "gpt-4-turbo", api_key: str = ""
) -> list[dict]:
    """
    Generates 5 fake superhero user accounts.

    Returns a list of dicts:
    [
        {
            "display_name": str,
            "provider": "Google" | "Cognito" | "AppleUser",
            "email": str
        },
        ...
    ]
    """

    client = OpenAI(api_key=api_key)

    prompt = (
        "Generate exactly 5 fake superhero user accounts.\n\n"
        "Requirements:\n"
        "- Use Marvel or Marvel-inspired superhero names\n"
        "- Each object must include:\n"
        "  - display_name (full hero name)\n"
        "  - provider (either 'Google' or 'Cognito' or 'AppleUser')\n"
        "  - email (must match the hero name, lowercase, simple domain like example.com or fantastic4.com)\n"
        "- Mix providers across users\n"
        "- Emails should be unique\n"
        "- Do NOT include real people\n\n"
        "Return ONLY valid JSON in the following format:\n"
        "[\n"
        "  {\n"
        '    "display_name": "Reed Richards",\n'
        '    "provider": "Google",\n'
        '    "email": "reed@fantastic4.com"\n'
        "  }\n"
        "]\n"
        "Do not include markdown, explanations, or extra text."
    )

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You generate realistic fake user account data "
                    "for a Marvel-style internal system."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0.7,
    )

    content = response.choices[0].message.content

    try:
        json_str = re.search(r"\[.*\]", content, re.DOTALL).group(0)
        return json.loads(json_str)
    except Exception as e:
        print("⚠️ Failed to parse JSON from GPT response:", e)
        print("Raw GPT output:\n", content)
        return []

In [4]:
def get_user_pool_id(stage: str, region: str = "us-west-2") -> str:
    name = f"FamHelpDesk-UserPool-{stage}"
    pools = boto3.client("cognito-idp", region_name=region).list_user_pools(
        MaxResults=60
    )["UserPools"]
    return next(p["Id"] for p in pools if p["Name"] == name)

In [5]:
def get_cloudformation_resource(resource: str, region: str = "us-west-2") -> str:
    """
    Retrieve the SNS notification topic ARN from CloudFormation exports.

    Args:
        stage: The stage name (e.g., "Testing", "Prod")
        region: AWS region (default: us-west-2)

    Returns:
        The SNS topic ARN

    Raises:
        ValueError: If the export is not found
    """
    client = boto3.client("cloudformation", region_name=region)
    response = client.list_exports()

    for export in response.get("Exports", []):
        if export["Name"] == resource:
            return export["Value"]

    raise ValueError(f"Export '{resource}' not found in CloudFormation")

In [6]:
def get_open_ai_api_key():
    secrets_client = boto3.client("secretsmanager")
    response = secrets_client.get_secret_value(SecretId="OpenAI")
    secret_data = json.loads(response["SecretString"])
    return secret_data["api_key"]

In [7]:
stage = "Prod"
table_name = f"FamHelpDesk-{stage}"
notification_queue_url = get_cloudformation_resource(
    f"FamHelpDesk-NotificationQueueUrl-{stage}"
)
notification_queue_url

'https://sqs.us-west-2.amazonaws.com/851753231474/FamHelpDesk-NotificationQueue-Prod'

In [8]:
user_helper = UserProfileHelper(
    stage=stage, table_name=table_name, notification_queue_url=notification_queue_url
)
family_helper = FamilyHelper(
    stage=stage, table_name=table_name, notification_queue_url=notification_queue_url
)
family_membership_helper = FamilyMembershipHelper(
    stage=stage, table_name=table_name, notification_queue_url=notification_queue_url
)
group_membership_helper = GroupMembershipHelper(
    stage=stage, table_name=table_name, notification_queue_url=notification_queue_url
)
group_helper = GroupHelper(
    stage=stage, table_name=table_name, notification_queue_url=notification_queue_url
)
queue_helper = QueueHelper(
    stage=stage, table_name=table_name, notification_queue_url=notification_queue_url
)
ticket_helper = TicketHelper(
    stage=stage, table_name=table_name, notification_queue_url=notification_queue_url
)
ticket_comment_helper = TicketCommentHelper(
    stage=stage, table_name=table_name, notification_queue_url=notification_queue_url
)

In [9]:
accounts = create_super_hero_accounts(api_key=get_open_ai_api_key())
accounts

[{'display_name': 'Reed Richards',
  'provider': 'Google',
  'email': 'reed@fantastic4.com'},
 {'display_name': 'Tony Stark',
  'provider': 'AppleUser',
  'email': 'tony@starkindustries.com'},
 {'display_name': 'Peter Parker',
  'provider': 'Cognito',
  'email': 'peter@webhead.com'},
 {'display_name': 'Natasha Romanoff',
  'provider': 'Google',
  'email': 'natasha@avengers.com'},
 {'display_name': 'Stephen Strange',
  'provider': 'AppleUser',
  'email': 'stephen@sanctumsanctorum.com'}]

In [10]:
def create_cognito_user(
    user_pool_id: str, user: dict, region: str = "us-west-2"
) -> tuple:
    client = boto3.client("cognito-idp", region_name=region)
    email = user.get("email")
    if not email:
        raise ValueError("User is missing email")

    username = email
    attributes = [
        {"Name": "email", "Value": email},
        {"Name": "email_verified", "Value": "true"},
    ]

    display_name = user.get("display_name")
    if display_name:
        attributes.append({"Name": "name", "Value": display_name})

    try:
        response = client.admin_create_user(
            UserPoolId=user_pool_id,
            Username=username,
            UserAttributes=attributes,
            MessageAction="SUPPRESS",
        )

        # Extract the sub from the response
        sub = None
        for attr in response["User"]["Attributes"]:
            if attr["Name"] == "sub":
                sub = attr["Value"]
                break

        temp_password = f"TestUser123"
        client.admin_set_user_password(
            UserPoolId=user_pool_id,
            Username=username,
            Password=temp_password,
            Permanent=True,
        )

        return username, sub

    except client.exceptions.UsernameExistsException:
        # If user exists, get their sub
        existing_user = client.admin_get_user(
            UserPoolId=user_pool_id, Username=username
        )
        sub = None
        for attr in existing_user["UserAttributes"]:
            if attr["Name"] == "sub":
                sub = attr["Value"]
                break

        return username, sub

In [11]:
user_pool_id = get_user_pool_id("Testing")

In [12]:
user_map = []
for user in accounts:
    _, sub = create_cognito_user(user_pool_id, user)
    user_helper.create_profile(
        user_id=sub,
        display_name=user["display_name"],
        provider=user["provider"],
        email=user["email"],
    )
    user_map.append({"user_id": sub, **user})

{"level":"INFO","location":"get_profile:134","message":"No user profile found for 78b19310-3081-70c2-4ad2-7871bc677ac0.","timestamp":"2026-02-12 21:12:25,535-0800","service":"service_undefined","taskName":"Task-2"}
{"level":"INFO","location":"create_profile:82","message":"Created user profile for 78b19310-3081-70c2-4ad2-7871bc677ac0","timestamp":"2026-02-12 21:12:25,579-0800","service":"service_undefined","taskName":"Task-2"}
{"level":"INFO","location":"get_settings:108","message":"Notification settings not found for user 78b19310-3081-70c2-4ad2-7871bc677ac0","timestamp":"2026-02-12 21:12:25,757-0800","service":"service_undefined","taskName":"Task-2"}
{"level":"INFO","location":"get_settings:108","message":"Notification settings not found for user 78b19310-3081-70c2-4ad2-7871bc677ac0","timestamp":"2026-02-12 21:12:25,783-0800","service":"service_undefined","taskName":"Task-2"}
{"level":"INFO","location":"create_default_settings:73","message":"Created default notification settings for u

In [14]:
family_name = "Avengers"
created_by = user_map[0]["user_id"]
family_description = "Asemble"
private = False

In [15]:
family = family_helper.create_family(
    family_name=family_name,
    created_by=created_by,
    family_description=family_description,
    private=private,
)

{"level":"INFO","location":"create_family:70","message":"Created family F4778122323","timestamp":"2026-02-12 21:13:29,365-0800","service":"service_undefined","taskName":"Task-2"}
{"level":"INFO","location":"create_family_audit_record:69","message":"Created family audit record for FAMILY F4778122323 action CREATE by user 78b19310-3081-70c2-4ad2-7871bc677ac0","timestamp":"2026-02-12 21:13:29,393-0800","service":"service_undefined","taskName":"Task-2"}
{"level":"INFO","location":"get_membership:55","message":"No membership for family F4778122323 and user 78b19310-3081-70c2-4ad2-7871bc677ac0.","timestamp":"2026-02-12 21:13:29,644-0800","service":"service_undefined","taskName":"Task-2"}
{"level":"INFO","location":"create_membership:133","message":"Created membership for user 78b19310-3081-70c2-4ad2-7871bc677ac0 in family F4778122323 (admin=True).","timestamp":"2026-02-12 21:13:29,697-0800","service":"service_undefined","taskName":"Task-2"}
{"level":"INFO","location":"create_family_audit_rec

In [16]:
family_id = family.family_id
family_id

'F4778122323'

In [17]:
for user in user_map[1:]:
    family_membership_helper.create_membership_request(
        family_id=family_id, user_id=user["user_id"]
    )

{"level":"INFO","location":"get_membership:55","message":"No membership for family F4778122323 and user 9811e3f0-20a1-707b-7cd8-e07fbaf10fec.","timestamp":"2026-02-12 21:18:05,545-0800","service":"service_undefined","taskName":"Task-2"}
{"level":"INFO","location":"create_membership_request:92","message":"Created membership request for user 9811e3f0-20a1-707b-7cd8-e07fbaf10fec in family F4778122323.","timestamp":"2026-02-12 21:18:05,591-0800","service":"service_undefined","taskName":"Task-2"}
{"level":"INFO","location":"create_family_audit_record:69","message":"Created family audit record for MEMBER 9811e3f0-20a1-707b-7cd8-e07fbaf10fec action CREATE by user 9811e3f0-20a1-707b-7cd8-e07fbaf10fec","timestamp":"2026-02-12 21:18:05,715-0800","service":"service_undefined","taskName":"Task-2"}
{"level":"INFO","location":"create_notification_async:312","message":"Sent notification to SQS: [Family Membership Request]","timestamp":"2026-02-12 21:18:05,947-0800","service":"service_undefined","task

In [18]:
admins = family_membership_helper.get_all_admins(family_id=family_id)

{"level":"INFO","location":"get_all_admins:70","message":"Found 1 admins in family F4778122323.","timestamp":"2026-02-12 21:18:06,874-0800","service":"service_undefined","taskName":"Task-2"}


In [19]:
family_membership_requests = family_membership_helper.get_pending_membership_requests(
    family_id=family_id
)

{"level":"INFO","location":"get_pending_membership_requests:195","message":"Found 6 pending requests in family F4778122323.","timestamp":"2026-02-12 21:18:08,338-0800","service":"service_undefined","taskName":"Task-2"}


In [20]:
for request in family_membership_requests:
    family_membership_helper.review_membership_request(
        family_id=family_id,
        admin_user_id=admins[0],
        target_user_id=request["user_id"],
        approve=True,
    )

{"level":"INFO","location":"review_membership_request:315","message":"Approved membership request for user 28c133f0-2081-709c-6302-e3ad261fabc0 in family F4778122323.","timestamp":"2026-02-12 21:18:19,259-0800","service":"service_undefined","taskName":"Task-2"}
{"level":"INFO","location":"create_family_audit_record:69","message":"Created family audit record for MEMBER 28c133f0-2081-709c-6302-e3ad261fabc0 action UPDATE by user 78b19310-3081-70c2-4ad2-7871bc677ac0","timestamp":"2026-02-12 21:18:19,289-0800","service":"service_undefined","taskName":"Task-2"}
{"level":"INFO","location":"create_notification_async:312","message":"Sent notification to SQS: [Family Membership Approved]","timestamp":"2026-02-12 21:18:19,328-0800","service":"service_undefined","taskName":"Task-2","notification_type":"Family Membership Approved","message_id":"446600e7-e86f-462e-8c76-85367339feb1","queue_url":"https://sqs.us-west-2.amazonaws.com/851753231474/FamHelpDesk-NotificationQueue-Prod"}
{"level":"INFO","lo

In [21]:
members = family_membership_helper.get_all_members(family_id=family_id)
members

{"level":"INFO","location":"get_all_members:209","message":"Found 7 active members in family F4778122323.","timestamp":"2026-02-12 21:18:48,117-0800","service":"service_undefined","taskName":"Task-2"}


[{'family_id': 'F4778122323',
  'user_id': '28c133f0-2081-709c-6302-e3ad261fabc0',
  'status': 'MEMBER',
  'is_admin': False,
  'request_date': 1770959886},
 {'family_id': 'F4778122323',
  'user_id': '38215360-90f1-7016-ad63-fb2c66cc137d',
  'status': 'MEMBER',
  'is_admin': False,
  'request_date': 1770959671},
 {'family_id': 'F4778122323',
  'user_id': '584133a0-9061-7040-725a-9059abc6b413',
  'status': 'MEMBER',
  'is_admin': False,
  'request_date': 1770959886},
 {'family_id': 'F4778122323',
  'user_id': '68f15370-e061-70f7-673f-428760655110',
  'status': 'MEMBER',
  'is_admin': False,
  'request_date': 1770959869},
 {'family_id': 'F4778122323',
  'user_id': '78b19310-3081-70c2-4ad2-7871bc677ac0',
  'status': 'MEMBER',
  'is_admin': True,
  'request_date': 1770959609},
 {'family_id': 'F4778122323',
  'user_id': '9811e3f0-20a1-707b-7cd8-e07fbaf10fec',
  'status': 'MEMBER',
  'is_admin': False,
  'request_date': 1770959885},
 {'family_id': 'F4778122323',
  'user_id': 'b80193e0-0071-7

In [22]:
groups_queues_map = [
    {
        "group": "Avengers",
        "queues": [
            "Global Threat Assessment",
            "Avengers Tower Operations",
            "Rapid Response Team",
            "Interagency Coordination",
        ],
    },
    {
        "group": "Illuminati",
        "queues": [
            "Confidential Briefings",
            "Multiversal Risk Review",
            "Artifact Containment",
            "Strategic Decisions",
        ],
    },
    {
        "group": "Future Foundation",
        "queues": [
            "Research & Development",
            "Youth Outreach Programs",
            "Experimental Technology",
            "Educational Initiatives",
        ],
    },
    {
        "group": "Defenders",
        "queues": [
            "Mystic Incidents",
            "Urban Crisis Support",
            "Unaligned Threats",
            "Emergency Consultations",
        ],
    },
    {
        "group": "Fantastic Force",
        "queues": [
            "Field Operations",
            "Special Assignments",
            "Containment Support",
            "Reconnaissance",
        ],
    },
    {
        "group": "Guardians of the Galaxy",
        "queues": [
            "Deep Space Missions",
            "Interstellar Diplomacy",
            "Ship Maintenance",
            "Cosmic Threat Reports",
        ],
    },
    {
        "group": "Alpha Flight",
        "queues": [
            "Northern Region Operations",
            "National Defense Requests",
            "Weather Anomalies",
            "Border Incident Response",
        ],
    },
    {
        "group": "S.W.O.R.D.",
        "queues": [
            "Orbital Surveillance",
            "Extraterrestrial Contact",
            "Space Station Operations",
            "Cosmic Intelligence Analysis",
        ],
    },
]

In [23]:
group_created_map = []
for highlighted_group in groups_queues_map:
    group_creator = random.choice(members)
    group = group_helper.create_group(
        family_id=family_id,
        group_name=highlighted_group["group"],
        created_by=group_creator["user_id"],
    )

    num_to_pick = random.randint(1, 3)
    memers_in_group = random.sample(members, num_to_pick)

    for user in memers_in_group:
        try:
            group_membership_helper.create_membership_request(
                family_id=family_id, group_id=group.group_id, user_id=user["user_id"]
            )
        except MembershipAlreadyExistsAsMember:
            pass
    group_membership_requests = group_membership_helper.get_pending_membership_requests(
        family_id=family_id, group_id=group.group_id
    )
    for request in group_membership_requests:
        group_membership_helper.review_membership_request(
            family_id=family_id,
            group_id=group.group_id,
            admin_user_id=group_creator["user_id"],
            target_user_id=request["user_id"],
            approve=True,
        )

    created_queues = []
    for queue in highlighted_group["queues"]:
        queue = queue_helper.create_queue(
            family_id=family_id,
            group_id=group.group_id,
            queue_name=queue,
            created_by=group_creator["user_id"],
        )
        created_queues.append(queue.queue_id)

    group_created_map.append({"id": group.group_id, "queues": created_queues})

{"level":"INFO","location":"create_group:77","message":"Created group G1666944911 in family F4778122323","timestamp":"2026-02-12 21:18:50,282-0800","service":"service_undefined","taskName":"Task-2"}
{"level":"INFO","location":"create_family_audit_record:69","message":"Created family audit record for GROUP G1666944911 action CREATE by user 9811e3f0-20a1-707b-7cd8-e07fbaf10fec","timestamp":"2026-02-12 21:18:50,317-0800","service":"service_undefined","taskName":"Task-2"}
{"level":"INFO","location":"create_queue:55","message":"Created queue Q2443595702 in family F4778122323","timestamp":"2026-02-12 21:18:50,509-0800","service":"service_undefined","taskName":"Task-2"}
{"level":"INFO","location":"create_family_audit_record:69","message":"Created family audit record for QUEUE Q2443595702 action CREATE by user 9811e3f0-20a1-707b-7cd8-e07fbaf10fec","timestamp":"2026-02-12 21:18:50,541-0800","service":"service_undefined","taskName":"Task-2"}
{"level":"INFO","location":"create_group:98","message"

In [24]:
group_created_map

[{'id': 'G1666944911',
  'queues': ['Q6837485469', 'Q8242751109', 'Q6716075407', 'Q2787348086']},
 {'id': 'G1144758797',
  'queues': ['Q5150423807', 'Q3334181080', 'Q5824136648', 'Q6251089181']},
 {'id': 'G9896033704',
  'queues': ['Q9215800433', 'Q6128025290', 'Q2215481497', 'Q3813492458']},
 {'id': 'G7429050171',
  'queues': ['Q4359796315', 'Q0596544280', 'Q9648857021', 'Q2772081421']},
 {'id': 'G7455749946',
  'queues': ['Q3479989369', 'Q6353243084', 'Q3109279731', 'Q8857097251']},
 {'id': 'G0240136256',
  'queues': ['Q2354252379', 'Q8601647176', 'Q5239454013', 'Q0159369056']},
 {'id': 'G3488546028',
  'queues': ['Q6736266993', 'Q3672189277', 'Q7366271655', 'Q1555717382']},
 {'id': 'G2533693381',
  'queues': ['Q8505889859', 'Q4337058515', 'Q7111303947', 'Q4329540529']}]

In [25]:
def generate_marvel_ticket_gpt(model: str = "gpt-4-turbo", api_key: str = "") -> dict:
    """
    Generates a fake Marvel-style ticket with title, description, and 0–5 comments.
    Returns a dict:
    {
        "title": str,
        "description": str,
        "comments": list[str]
    }
    """

    client = OpenAI(api_key=api_key)

    prompt = (
        "Create a fake Marvel-universe support ticket.\n\n"
        "Requirements:\n"
        "- Theme it like Avengers, Fantastic Four, or cosmic Marvel operations\n"
        "- The issue should sound operational or incident-based\n"
        "- Tone should be semi-serious, like an internal ticketing system\n"
        "- Comments represent internal updates or replies\n"
        "- Do not add the super hero's name who made the commeng\n"
        "- Number of comments must be between 0 and 5\n\n"
        "Return ONLY valid JSON in the following format:\n"
        "{\n"
        '  "title": "string",\n'
        '  "description": "string",\n'
        '  "comments": ["string", "string"]\n'
        "}\n"
        "Do not include markdown, explanations, or extra text."
    )

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You generate realistic internal ticket data "
                    "for a Marvel-style operations system."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0.8,
    )

    content = response.choices[0].message.content

    try:
        json_str = re.search(r"\{.*\}", content, re.DOTALL).group(0)
        return json.loads(json_str)
    except Exception as e:
        print("⚠️ Failed to parse JSON from GPT response:", e)
        print("Raw GPT output:\n", content)
        return {}

In [26]:
severities = [1.0, 2.0, 2.5, 3.0, 4.0, 5.0]

In [28]:
for _ in range(10):
    group_queue = random.choice(group_created_map)
    group = group_queue["id"]
    queue = random.choice(group_queue["queues"])
    assigned_to = random.choice(members + [None])
    print(group, queue, assigned_to)
    generated_info = generate_marvel_ticket_gpt(api_key=get_open_ai_api_key())
    print(generated_info)

    ticket = ticket_helper.create_ticket(
        family_id=family_id,
        group_id=group,
        queue_id=queue,
        title=generated_info["title"],
        severity=random.choice(severities),
        created_by=random.choice(members)["user_id"],
        description=generated_info["description"],
        assigned_to=assigned_to["user_id"] if assigned_to is not None else None,
    )
    time.sleep(random.uniform(1, 10))

    for comment in generated_info["comments"]:
        ticket_comment_helper.create_comment(
            ticket_id=ticket.ticket_id,
            comment_user=random.choice(members)["user_id"],
            comment_body=comment,
        )
        time.sleep(random.uniform(1, 10))

G1666944911 Q8242751109 {'family_id': 'F4778122323', 'user_id': '68f15370-e061-70f7-673f-428760655110', 'status': 'MEMBER', 'is_admin': False, 'request_date': 1770959869}
{'title': 'Quantum Tunnel Malfunction', 'description': 'Urgent: The quantum tunnel is currently offline due to an unknown energy flux. This is preventing any form of quantum research or travel. Immediate assistance required to diagnose and resolve the issue. Please prioritize as this affects multiple ongoing operations.', 'comments': ['Initial diagnostics seem to suggest a disruption in the power supply line directly linked to the quantum cores. Suggest checking the power configurations and circuit integrity.', 'I have rerouted the auxiliary power and am initiating a cold reboot of the system. Should know more in 10 minutes.', 'The reboot has not resolved the issue. It appears there might be a deeper hardware issue. Requesting a full hardware inspection.', 'Hardware team dispatched and on site. Preliminary inspection 

In [ ]:
user_ids = [account["user_id"] for account in user_map]

In [ ]:
user_ids

In [ ]:
family_id